In [1]:
from pathlib import Path
import random
import re
import json
import os

import pandas as pd
from lxml import etree
from tqdm.auto import tqdm



#1. Clone Repository (we need the dataset)



In [2]:
!git clone https://github.com/LedionaLame95/IR-CW2.git

Cloning into 'IR-CW2'...
remote: Enumerating objects: 104935, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 104935 (delta 2), reused 22 (delta 1), pack-reused 104912 (from 2)
Receiving objects: 100% (104935/104935), 770.18 MiB | 17.12 MiB/s, done.
Resolving deltas: 100% (4792/4792), done.
Updating files: 100% (52052/52052), done.


In [3]:
os.chdir('./IR-CW2')

In [4]:
!pwd
!ls

/content/IR-CW2
data  notebooks  parser.py  requirements.txt  results


# 2. Dataset Parsing

In [5]:
# set project root automatically if notebook is inside /notebooks
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

#dataset path
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "clef_ip"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "results" / "demo"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# mvp settings
SUBSET_SIZE = 2000          # start small increase later
USE_RANDOM_SAMPLE = False   # False = first N sorted files, True = random sample
RANDOM_SEED = 42
TOP_K = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR exists:", RAW_DIR.exists())
print("INTERIM_DIR:", INTERIM_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_ROOT: /content/IR-CW2
RAW_DIR exists: True
INTERIM_DIR: /content/IR-CW2/data/interim
RESULTS_DIR: /content/IR-CW2/results/demo


In [6]:
xml_files = sorted(RAW_DIR.rglob("*.xml"))

if len(xml_files) == 0:
    raise FileNotFoundError(
        f"No XML files found under {RAW_DIR}. Put your CLEF-IP XML files in data/raw/clef_ip/"
    )

if USE_RANDOM_SAMPLE:
    rng = random.Random(RANDOM_SEED)
    subset_files = rng.sample(xml_files, min(SUBSET_SIZE, len(xml_files)))
else:
    subset_files = xml_files[:min(SUBSET_SIZE, len(xml_files))]

manifest_path = INTERIM_DIR / "mvp_subset_files.txt"
manifest_path.write_text("\n".join(str(p) for p in subset_files), encoding="utf-8")

print(f"Total XML files found: {len(xml_files):,}")
print(f"MVP subset size: {len(subset_files):,}")
print(f"Subset manifest saved to: {manifest_path}")

subset_files[:5]

Total XML files found: 52,040
MVP subset size: 2,000
Subset manifest saved to: /content/IR-CW2/data/interim/mvp_subset_files.txt


[PosixPath('/content/IR-CW2/data/raw/clef_ip/WO-1979000001-A1.xml'),
 PosixPath('/content/IR-CW2/data/raw/clef_ip/WO-1979000002-A1.xml'),
 PosixPath('/content/IR-CW2/data/raw/clef_ip/WO-1979000005-A1.xml'),
 PosixPath('/content/IR-CW2/data/raw/clef_ip/WO-1979000007-A1.xml'),
 PosixPath('/content/IR-CW2/data/raw/clef_ip/WO-1979000008-A1.xml')]

In [7]:
TAG_SPACE_RE = re.compile(r"\s+")

def clean_join(text_items):
    """
    joins text fragments, strips whitespace, and normalises spacing
    """
    parts = []
    for item in text_items:
        if item is None:
            continue
        text = str(item).strip()
        if text:
            parts.append(text)
    joined = " ".join(parts)
    joined = TAG_SPACE_RE.sub(" ", joined).strip()
    return joined

def first_nonempty(root, xpath_list):
    """
    tries multiple xpath patterns and returns the first non-empty result
    """
    for xp in xpath_list:
        try:
            result = root.xpath(xp)
            if isinstance(result, list):
                text = clean_join(result)
            else:
                text = str(result).strip()
            if text:
                return text
        except Exception:
            continue
    return ""

def parse_patent_xml(xml_path: Path):
    """
    robust patent xml parser for mvp use
    extracts a few key fields with fallbacks
    """
    parser = etree.XMLParser(recover=True, huge_tree=True)
    tree = etree.parse(str(xml_path), parser)
    root = tree.getroot()

    doc_id = first_nonempty(root, [
        'string(//*[local-name()="publication-reference"]//*[local-name()="doc-number"][1])',
        'string(//*[local-name()="document-id"]//*[local-name()="doc-number"][1])',
        'string(//*[@doc-number][1]/@doc-number)',
        'string(//*[@id][1]/@id)'
    ])
    if not doc_id:
        doc_id = xml_path.stem

    title = first_nonempty(root, [
        '//*[local-name()="invention-title"]/text()',
        '//*[local-name()="title"]/text()'
    ])

    abstract = first_nonempty(root, [
        '//*[local-name()="abstract"]//text()'
    ])

    claims = first_nonempty(root, [
        '//*[local-name()="claims"]//text()',
        '//*[local-name()="claim"]//text()'
    ])

    description = first_nonempty(root, [
        '//*[local-name()="description"]//text()'
    ])

    ipc = first_nonempty(root, [
        '//*[local-name()="classification-ipc"]//text()'
    ])

    if not any([title, abstract, claims, description]):
        raise ValueError("No usable text extracted from XML")

    return {
        "doc_id": doc_id,
        "title": title,
        "abstract": abstract,
        "claims": claims,
        "description": description,
        "ipc": ipc,
        "source_file": str(xml_path)
    }

In [8]:
records = []
failures = []

for xml_path in tqdm(subset_files, desc="parsing xml"):
    try:
        records.append(parse_patent_xml(xml_path))
    except Exception as e:
        failures.append({"file": str(xml_path), "error": repr(e)})

#parsed df creation
docs_df = pd.DataFrame(records)

if docs_df.empty:
    raise ValueError("Parsing produced no usable patent records.")

# build the searchable text for the first MVP baseline
docs_df["search_text"] = (
    docs_df["title"].fillna("") + " " + docs_df["abstract"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

docs_df = docs_df[docs_df["search_text"] != ""].copy()
docs_df = docs_df.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)

parsed_output_path = INTERIM_DIR / "parsed_patents_mvp.jsonl"
docs_df.to_json(parsed_output_path, orient="records", lines=True, force_ascii=False)

failures_path = INTERIM_DIR / "parse_failures_mvp.json"
failures_path.write_text(json.dumps(failures, indent=2), encoding="utf-8")

print(f"Parsed records kept: {len(docs_df):,}")
print(f"Parsing failures: {len(failures):,}")
print(f"Parsed records saved to: {parsed_output_path}")
print(f"Failure log saved to: {failures_path}")

parsing xml:   0%|          | 0/2000 [00:00<?, ?it/s]

Parsed records kept: 2,000
Parsing failures: 0
Parsed records saved to: /content/IR-CW2/data/interim/parsed_patents_mvp.jsonl
Failure log saved to: /content/IR-CW2/data/interim/parse_failures_mvp.json


In [9]:
docs_df.head(10)

,doc_id,title,abstract,claims,description,ipc,source_file,search_text
0,1979000001,APPARATUS FOR DETERMINING THE EPIDERMIC GROUP ...,,REVENDICATIONS 1) Dispositif permettant la déf...,La présente invention constitue un dispositif ...,2 A61B 5/00,/content/IR-CW2/data/raw/clef_ip/WO-1979000001...,APPARATUS FOR DETERMINING THE EPIDERMIC GROUP ...
1,1979000002,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORESIS,,CLAIMS 1. An electrophoresis membrane of known...,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORE...,2 G01N 27/26 B01D 13/02,/content/IR-CW2/data/raw/clef_ip/WO-1979000002...,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORESIS
2,1979000005,IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER,,"WHAT IS CLAIMED IS: "" 1. A mechanism for a rot...",IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER ...,2 F16D 67/02,/content/IR-CW2/data/raw/clef_ip/WO-1979000005...,IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER
3,1979000007,SELF-RELEASABLE FISHING SINKER HOLDING DEVICE,A device for releasable holding fishing sinker...,Patentansprüche . Behälter für einen Wurfkörpe...,Beschreibung Selbstlösende Wurfkörperbefestigu...,2 A01K 91/02,/content/IR-CW2/data/raw/clef_ip/WO-1979000007...,SELF-RELEASABLE FISHING SINKER HOLDING DEVICE ...
4,1979000008,A DEVICE FOR ABSORBING URINE WITH INCONTINENT ...,,"CLAIMS ""11 _ 1. A device for absorbing urine w...",A device for Absorbing Urine with Incontinent ...,2 A61F 5/44 A61G 9/00,/content/IR-CW2/data/raw/clef_ip/WO-1979000008...,A DEVICE FOR ABSORBING URINE WITH INCONTINENT ...
5,1979000009,METHOD AND APPARATUS FOR CARRYING OUT CHEMICAL...,,"CLAIMS 1. A method of carrying out, in at leas...",METHOD AND APPARATUS FOR CARRYING OUT CHEMICAL...,2 B01J 8/26 C10J 3/46 F23D 19/00 B01D 53/12,/content/IR-CW2/data/raw/clef_ip/WO-1979000009...,METHOD AND APPARATUS FOR CARRYING OUT CHEMICAL...
6,1979000010,PLURAL SENSOR ENDS DOWN DETECTING APPARATUS,,THAT WHICH IS CLAIMED IS; 1. In the combinatio...,PLURAL SENSOR ENDS DOWN DETECTING APPARATUS Te...,2 D01H 13/16,/content/IR-CW2/data/raw/clef_ip/WO-1979000010...,PLURAL SENSOR ENDS DOWN DETECTING APPARATUS
7,1979000011,SILENT RUNNING INTERNAL COMBUSTION MOTOR POWER...,,Claims 1. Silent-running motor power unit with...,Silent-running; internal combustion motor powe...,2 F01N 7/00,/content/IR-CW2/data/raw/clef_ip/WO-1979000011...,SILENT RUNNING INTERNAL COMBUSTION MOTOR POWER...
8,1979000012,AUTOMATIC CONTROL OF THE CLOSING STRENGTH NEED...,,Patentansorüche: 1. Selbsttätige Schließkraftr...,Selbsttätige Schließkra tregelung zur Dichthai...,2 G05D 16/20 B01D 25/12,/content/IR-CW2/data/raw/clef_ip/WO-1979000012...,AUTOMATIC CONTROL OF THE CLOSING STRENGTH NEED...
9,1979000014,BIOLOGICALLY COMPATIBLE TAMPON SPONGE,,1. A method for forming a biocompatible tampon...,BIOLOGICALLY COMPATIBLE TAMPON SPONGE Backgrou...,2 C08J 9/16 A61F 5/46 B01D 47/16 B32B 5/18,/content/IR-CW2/data/raw/clef_ip/WO-1979000014...,BIOLOGICALLY COMPATIBLE TAMPON SPONGE


In [10]:
sample_row = docs_df.iloc[0]

print("DOC ID:")
print(sample_row["doc_id"])
print("\nTITLE:")
print(sample_row["title"][:500])
print("\nABSTRACT:")
print(sample_row["abstract"][:1000])
print("\nCLAIMS:")
print(sample_row["claims"][:1000])
print("\nDESCRIPTION:")
print(sample_row["description"][:1200])
print("\nSEARCH TEXT:")
print(sample_row["search_text"][:1000])

DOC ID:
1979000001

TITLE:
APPARATUS FOR DETERMINING THE EPIDERMIC GROUP OF A LIVING BEING:DRY-PART DRY-PART GREASY-GREASY

ABSTRACT:


CLAIMS:
REVENDICATIONS 1) Dispositif permettant la définition d'un groupe epidermique, soit : Sec - Mixte Sec - Mixte Gras - Gras d'un être vivant, pour ce faire en chargeant dans un premier temps celui-ci d'un potentiel énergétique et en le déchargeant dans un deuxième temps à travers sa résistance épidermi- que ; celle-ci étant enregistrée et mesurée à l'aide d'un contrôleur énergétique approprié, définissant ainsi le groupe epidermique du sujet. . 2) Dispositif selon la revendication 1, caractérisé d'un dispo¬ sitif d'ionisation des cellules d'un être vivant, comportant un organe de préhension, ledit dispositif générant par lui-même une énergie ionisante par effet électrolytique à l'aide de deux électrodes en métal conductible, en contact chacune dans un main du sujet à ioniser, en faisant interférer l'énergie organique de celui-ci, laquelle ferme l

#3. Dataframe Chunking

In [11]:
def create_overlapping_chunks(text, title, chunk_size=300, overlap=50):

    """Breaks text into overlapping chunks, prepending the Title for context."""
    if not text or not str(text).strip():
        return []

    words = str(text).split()
    chunks = []
    step_size = chunk_size - overlap

    for i in range(0, len(words), step_size):
        chunk_words = words[i : i + chunk_size]
        chunk_text = " ".join(chunk_words)

        # Inject context!
        contextualized_chunk = f"Title: {title} | Text: {chunk_text}"
        chunks.append(contextualized_chunk)

        if i + chunk_size >= len(words):
            break

    return chunks

In [12]:
all_chunks = []
chunk_metadata = []

# Iterate over the rows of the pandas DataFrame
for idx, row in tqdm(docs_df.iterrows(), total=len(docs_df), desc="Chunking patents"):

    # 1. Combine the heavy text fields
    full_text = f"{row.get('abstract', '')} {row.get('description', '')} {row.get('claims', '')}"

    # Clean up any massive whitespace gaps
    full_text = TAG_SPACE_RE.sub(" ", full_text).strip()

    title = row.get('title', 'Unknown Title')
    if pd.isna(title):
        title = "Unknown Title"

    # 2. Create the chunks
    chunks = create_overlapping_chunks(full_text, title=title)

    # 3. Store them with metadata tracking
    for chunk_id, chunk_text in enumerate(chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append({
            "doc_id": row["doc_id"],
            "chunk_id": chunk_id,
            "text": chunk_text
        })

print(f"\nOriginal patents: {len(docs_df):,}")
print(f"Total chunks generated for FAISS: {len(all_chunks):,}")
assert len(all_chunks) == len(chunk_metadata)

Chunking patents:   0%|          | 0/2000 [00:00<?, ?it/s]


Original patents: 2,000
Total chunks generated for FAISS: 54,391


#4. Offline Indexing (Facebook AI Similarity Search - FAISS)

In [13]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.3 MB/s eta 0:00:00


In [14]:
# 4. Dense Embedding and FAISS Indexing

import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# 1. Load the Model
MODEL_NAME = "anferico/bert-for-patents"
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# 2. Embed the Chunks
print(f"Embedding {len(all_chunks):,} chunks...")
embeddings = model.encode(
    all_chunks,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True # Crucial for Cosine Similarity
)

# 3. Create the FAISS Index
print("Initializing FAISS IndexFlatIP...")
embedding_dim = model.get_sentence_embedding_dimension()
index = faiss.IndexFlatIP(embedding_dim)

print("Adding vectors to FAISS...")
index.add(np.array(embeddings, dtype=np.float32))

# 4. Save to your INTERIM_DIR
faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
meta_path = INTERIM_DIR / "chunk_metadata_mvp.pkl"

print(f"Saving FAISS index to {faiss_path}...")
faiss.write_index(index, str(faiss_path))

print(f"Saving metadata to {meta_path}...")
with open(meta_path, "wb") as f:
    pickle.dump(chunk_metadata, f)

print("Offline Indexing Complete!")

Loading anferico/bert-for-patents...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.38G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: anferico/bert-for-patents
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

Embedding 54,391 chunks...


Batches:   0%|          | 0/213 [00:00<?, ?it/s]

Initializing FAISS IndexFlatIP...
Adding vectors to FAISS...
Saving FAISS index to /content/IR-CW2/data/interim/patent_dense_mvp.index...
Saving metadata to /content/IR-CW2/data/interim/chunk_metadata_mvp.pkl...
Offline Indexing Complete!


#5. Online Search

In [15]:
# 5. Online Search Function (MaxP Aggregation)

def search_dense_index(query, top_n=10, faiss_k=100):
    """
    Embeds a query, searches FAISS, and aggregates chunk scores to rank the parent patents.
    """
    # 1. Load assets from your specific directories
    faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
    meta_path = INTERIM_DIR / "chunk_metadata_mvp.pkl"

    if not faiss_path.exists() or not meta_path.exists():
        raise FileNotFoundError("FAISS index or metadata not found. Run the indexing cell first!")

    index = faiss.read_index(str(faiss_path))
    with open(meta_path, "rb") as f:
        metadata = pickle.load(f)

    # 2. Embed the query
    query_embedding = model.encode([query], normalize_embeddings=True)

    # 3. Search the index
    # We ask for faiss_k (e.g., 100 chunks) to ensure we get enough unique patents
    distances, indices = index.search(np.array(query_embedding, dtype=np.float32), k=faiss_k)

    # 4. Aggregate Scores via MaxP (Maximum Passage)
    patent_scores = {}

    for score, idx in zip(distances[0], indices[0]):
        chunk_meta = metadata[idx]
        doc_id = chunk_meta['doc_id']

        # MaxP Logic: Update score if we haven't seen this patent,
        # or if this specific chunk has a higher score.
        if doc_id not in patent_scores or score > patent_scores[doc_id]['score']:
            patent_scores[doc_id] = {
                'score': float(score),
                'best_chunk_text': chunk_meta['text']
            }

    # 5. Sort by aggregated score descending
    ranked_patents = sorted(
        patent_scores.items(),
        key=lambda item: item[1]['score'],
        reverse=True
    )

    # Return only the number of unique patents requested
    return ranked_patents[:top_n]

#5.a) Free Text Search

In [16]:
test_query = "A device with an absorption core made of cellulose wadding for absorbing liquids"

print(f"User Query: '{test_query}'")
print("=" * 70)

# Run the search and fetch the top 5 patents
results = search_dense_index(test_query, top_n=5, faiss_k=100)

if not results:
    print("No results found. Check if your FAISS index was populated correctly.")
else:
    for rank, (doc_id, data) in enumerate(results, 1):
        print(f"Rank {rank} | Patent ID: {doc_id} | MaxP Score: {data['score']:.4f}")

        # Print a snippet of the matching chunk to see exactly *why* it matched
        matching_text = data['best_chunk_text']

        # Clean up the snippet for display (truncate if it's too long)
        snippet = matching_text[:250] + "..." if len(matching_text) > 250 else matching_text
        print(f"Matched Text: {snippet}")
        print("-" * 70)

User Query: 'A device with an absorption core made of cellulose wadding for absorbing liquids'
Rank 1 | Patent ID: 1979000008 | MaxP Score: 0.7189
Matched Text: Title: A DEVICE FOR ABSORBING URINE WITH INCONTINENT PERSONS | Text: that the absorption core (9) of the diaper consists of fluffed cellulose wadding, and in that the liquid-perme¬ able material (lθ) consists of non-woven fabric, at least on the side...
----------------------------------------------------------------------
Rank 2 | Patent ID: 2000000115 | MaxP Score: 0.6989
Matched Text: Title: URINE COLLECTOR COLLECTEUR D'URINE | Text: A urine management device (10) according to claim 1 , wherein said wall material comprises 3 layers, wherein said inner surface (18) is a nonwoven layer, said outer surface (17) is said fibrous hydrop...
----------------------------------------------------------------------
Rank 3 | Patent ID: 2000000112 | MaxP Score: 0.6954
Matched Text: Title: A METHOD FOR COLLECTING AND DISPOSING OF HUMAN WAS

#5.b) Document to Document Search

In [17]:
def construct_query_from_document(query_patent, strategy="chunking"):
    """
    Extracts the query from a patent document based on the specified strategy.
    Always returns a List[str]
    """
    assert strategy in ["first_page", "chunking"], "Unknown strategy"

    if strategy == "first_page":
        # Strategy 1: Just concatenate Title and Abstract (returns a list of 1)
        text = f"{query_patent.get('title', '')} {query_patent.get('abstract', '')}"
        return [text.strip()]

    elif strategy == "chunking":
        # Strategy 2: The full chunk-to-chunk approach (returns a list of N)
        full_text = f"{query_patent.get('abstract', '')} {query_patent.get('description', '')} {query_patent.get('claims', '')}"
        full_text = TAG_SPACE_RE.sub(" ", full_text).strip()
        title = query_patent.get('title', 'Unknown Title')

        return create_overlapping_chunks(full_text, title)

In [18]:
def search_by_patent_document(query_patent, docs_df, strategy="chunking", top_n=10, faiss_k=100):

    # 1. Construct the query list dynamically
    query_chunks = construct_query_from_document(query_patent, strategy=strategy)
    if not query_chunks:
        return []

    # Extract IPC for post-filtering
    raw_ipc = str(query_patent.get('ipc', ''))
    target_ipc = raw_ipc[:4] if len(raw_ipc) >= 4 else None
    print(f"Input Patent Query has the following IPC: {target_ipc}")

    # 2. Load assets (In production, load these outside the function)
    faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
    meta_path = INTERIM_DIR / "chunk_metadata_mvp.pkl"
    index = faiss.read_index(str(faiss_path))
    with open(meta_path, "rb") as f:
        metadata = pickle.load(f)

    # 3. Embed the queries (Works exactly the same for 1 chunk or 50 chunks)
    query_embeddings = model.encode(query_chunks, normalize_embeddings=True, show_progress_bar=False)

    # 4. Search FAISS
    distances, indices = index.search(np.array(query_embeddings, dtype=np.float32), k=faiss_k)

    # 5. Global MaxP Aggregation
    patent_scores = {}

    for chunk_idx in range(len(query_chunks)):
        for score, idx in zip(distances[chunk_idx], indices[chunk_idx]):

            chunk_meta = metadata[idx]
            doc_id = chunk_meta['doc_id']

            # Prevent the query patent from retrieving itself
            if doc_id == query_patent['doc_id']:
                continue

            if (doc_id not in patent_scores) or (score > patent_scores[doc_id]['score']):
                patent_scores[doc_id] = {
                    'score': float(score),
                    'best_db_chunk': chunk_meta['text'],
                    'matched_query_chunk': query_chunks[chunk_idx]
                }

    # 6. Sort and Hydrate (with IPC Filtering)
    ranked_candidates = sorted(patent_scores.items(), key=lambda item: item[1]['score'], reverse=True)
    final_results = []

    for doc_id, data in ranked_candidates:
        if len(final_results) >= top_n:
            break

        matching_row = docs_df[docs_df['doc_id'] == doc_id]
        if matching_row.empty:
            continue

        candidate_patent = matching_row.iloc[0]
        candidate_ipc = str(candidate_patent['ipc'])

        # IPC Filter
        #if target_ipc and not candidate_ipc.startswith(target_ipc):
        #    continue

        final_results.append({
            'doc_id': doc_id,
            'score': data['score'],
            'title': candidate_patent['title'],
            'ipc': candidate_ipc,
            'match_reason_db': data['best_db_chunk']
        })

    return final_results

In [19]:
# For testing, we'll just grab a random patent from our existing DataFrame
sample_input_patent = docs_df.iloc[42]

doc_results = search_by_patent_document(sample_input_patent, docs_df=docs_df, top_n=5)

for rank, res in enumerate(doc_results, 1):
    print(f"Rank {rank} | ID: {res['doc_id']} | Score: {res['score']:.4f} | IPC: {res['ipc']}")
    print(f"Title: {res['title']}")
    print("-" * 50)

Input Patent Query has the following IPC: 2 B4
Rank 1 | ID: 2000001043 | Score: 0.9825 | IPC: 7 H01R 43/16 H01H 11/04
Title: METHOD FOR MAKING A CONTACT ELEMENT PROCEDE DE REALISATION D'UNE PIECE CONTACTEE
--------------------------------------------------
Rank 2 | ID: 1979000041 | Score: 0.9815 | IPC: 2 A63B 3/00
Title: APPLIANCE FOR ASYMMETRICAL BARS FOR SPORT'S USE
--------------------------------------------------
Rank 3 | ID: 1979000170 | Score: 0.9815 | IPC: 2 B65D 5/56
Title: MANUFACTURING PROCESS OF A FOLDED CARDBOARD PACKING COVERED WITH AN IMPERVIOUS SHEET AND PACKING OBTAINED FROM SUCH PROCESS
--------------------------------------------------
Rank 4 | ID: 1979000171 | Score: 0.9811 | IPC: 2 B65D 5/56
Title: PACKING COMPRISING IN ASSOCIATION ONE PART OF CARDBOARD AND ONE PART OF SYNTHETIC MATERIAL
--------------------------------------------------
Rank 5 | ID: 1979000065 | Score: 0.9809 | IPC: 2 A61B 6/14
Title: APPARATUS FOR PANORAMIC RADIOGRAPHY
---------------------------